# MLM Pretraining and Concept Masking for RuModernBERT

This notebook performs two types of additional pretraining for RuModernBERT:

1. **Standard MLM (Masked Language Modeling)**  
   We train the model on raw text from factRuEval‑2016 to improve domain adaptation.

2. **NER Concept Masking**  
   We mask tokens belonging to named entities (PER / ORG / LOC) and train the model to reconstruct them.  
   This encourages the model to learn entity‑specific contextual patterns.

Both approaches aim to improve downstream NER performance before fine‑tuning.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

## Prepare raw text for MLM

We reconstruct text from tokens and create a dataset of plain strings.

In [ ]:
def reconstruct_text(example):
    text, _ = tokens_to_text_and_offsets(example["tokens"])
    return {"text": text}

mlm_ds = ds_small.map(reconstruct_text)
mlm_ds = mlm_ds.remove_columns(["tokens", "ner_tags", "ner_tags_str", "length", "id"])

## Tokenization for MLM

In [ ]:
model_id = "deepvk/RuModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_mlm(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=192
    )

tokenized_mlm = mlm_ds.map(tokenize_mlm, batched=True)

## Standard MLM Pretraining

In [ ]:
mlm_model = AutoModelForMaskedLM.from_pretrained(model_id).to(DEVICE)

mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15
)

mlm_args = TrainingArguments(
    output_dir="./mlm_pretraining",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=tokenized_mlm["train"],
    data_collator=mlm_collator
)

In [ ]:
mlm_trainer.train()

# NER Concept Masking

We mask tokens belonging to named entities (PER / ORG / LOC)  
and train the model to reconstruct them.

This encourages the model to learn entity‑specific contextual patterns.

In [ ]:
from src.modernbert.concept_masking_collator import NERConceptMaskCollator

concept_collator = NERConceptMaskCollator(tokenizer, mask_prob=0.15)

concept_model = AutoModelForMaskedLM.from_pretrained(model_id).to(DEVICE)

concept_args = TrainingArguments(
    output_dir="./concept_masking",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50
)

concept_trainer = Trainer(
    model=concept_model,
    args=concept_args,
    train_dataset=tokenized_mlm["train"],
    data_collator=concept_collator
)

In [ ]:
concept_trainer.train()

## Summary

We performed two types of additional pretraining:

- Standard MLM  
- NER Concept Masking  

Both checkpoints will be used in the next notebook for NER fine‑tuning.